In [ ]:
import pymrio
import pandas as pd
import os

# DB_PATH = 'D:/Programming/databases/IOT_2024_ixi'
DB_PATH = 'C:/Users/rusai1/bw_notebooks/data/exiobase/IOT_2022_ixi'

exio = pymrio.parse_exiobase3(path=DB_PATH)
data_path = DB_PATH


In [ ]:
exio.calc_all()

In [ ]:
type(exio.extensions)
type(exio)
exio.extensions

In [ ]:
print(exio.water.DataFrames)
print(exio.air_emissions.DataFrames)
print(exio.DataFrames)

In [ ]:
 # --- Explore available sectors (region ×   sector MultiIndex) ---
sectors = exio.Z.index.get_level_values("sector").unique()
regions = exio.Z.index.get_level_values("region").unique()

print(f"Regions ({len(regions)}): {list(regions)}")
print(f"\nTotal unique sectors: {len(sectors)}")
print("\nAll sectors:")
for i, s in enumerate(sectors):
    print(f"  {i:3d}  {s}")


In [ ]:
# --- Define custom final demand matching exio.Y structure ---
all_sectors = exio.Z.index.get_level_values("sector").unique()
sector_130 = all_sectors[130]
# sector_134 = all_sectors[134]

print(f"Sector 130: {sector_130}")
# print(f"Sector 134: {sector_134}")

# Mirror exio.Y exactly: same row and column MultiIndex, all zeros
y_custom = pd.DataFrame(0.0, index=exio.Y.index, columns=exio.Y.columns)

y_custom.loc[("SE", sector_130), ("SE", "Final consumption expenditure by households")] = 1.0
# y_custom.loc[("SE", sector_134), ("SE", "Final consumption expenditure by households")] = 1.0

# Verify: show only non-zero rows in the household column
col = ("SE", "Final consumption expenditure by households")
print("\ny_custom non-zero entries:")
print(y_custom.loc[y_custom[col] > 0, [col]])


In [ ]:
# --- Total output required to satisfy custom demand ---
# L [industries × industries] @ y_custom [industries × demand_cols]
# x_custom = exio.L @ y_custom
# print(f"x_custom shape: {x_custom.shape}")

# --- Air emission impacts ---
# S [stressors × industries] @ x_custom [industries × demand_cols] → [stressors × demand_cols]
air_impacts = exio.air_emissions.M @ y_custom

# Collapse to a single series (only one demand column is non-zero)
col = ("SE", "Final consumption expenditure by households")
air_impacts_series = air_impacts[col].sort_values(ascending=False)

print(f"Stressors: {len(air_impacts_series)}")
print("\nTop emitting stressors (kg):")
print(air_impacts_series[air_impacts_series > 0].to_frame("kg"))


In [ ]:
print("Columns match index:", (exio.L.columns == y_custom.index).all())
print("L.columns sample:", exio.L.columns[:2].tolist())
print("y_custom.index sample:", y_custom.index[:2].tolist())

In [ ]:
print(exio.air_emissions.M.columns.names)
print(y_custom.index.names)
print(air_impacts_series.isna().sum(), "NaNs out of", len(air_impacts_series))
print("NaN in L:", exio.L.isna().sum().sum())
print("NaN in S:", exio.air_emissions.S.isna().sum().sum())

In [ ]:
L_clean = exio.L.fillna(0)
S_clean = exio.air_emissions.S.fillna(0)

x_custom = L_clean @ y_custom
air_impacts = S_clean @ x_custom

col = ("SE", "Final consumption expenditure by households")
air_impacts_series = air_impacts[col].sort_values(ascending=False)
print(air_impacts_series[air_impacts_series > 0].to_frame("kg"))


In [ ]:
# Sum x_custom across demand columns → Series indexed by (region, sector)
x_vec = x_custom.sum(axis=1)

# D_cba: multiply each column of S_clean by the corresponding x value
D_cba_custom = S_clean.multiply(x_vec, axis=1)

# Total footprint per stressor
footprint_total = D_cba_custom.sum(axis=1).sort_values(ascending=False)
print(footprint_total[footprint_total > 0].to_frame("kg"))


In [ ]:
# # 1. Top upstream industries — summed across all air emission stressors
# upstream_by_industry = D_cba_custom.sum(axis=0).sort_values(ascending=False)
# print("Top 15 upstream industries (all stressors, kg):")
# print(upstream_by_industry.head(15).to_frame("kg"))

In [ ]:
# Identify GHG stressors in air_emissions
ghg_keywords = ['CO2', 'CH4', 'N2O', 'SF6', 'HFC', 'PFC', 'NF3']
ghg_stressors = [s for s in exio.air_emissions.S.index
               if any(k in s for k in ghg_keywords)]

print(f"GHG stressors ({len(ghg_stressors)}):")
for s in ghg_stressors:
  print(f"  {s}")


In [ ]:
A_clean = exio.A.fillna(0)

# numpy for efficient repeated matrix-vector products in the loop
A_np = A_clean.to_numpy()               # (7987 × 7987)
S_np = S_clean.to_numpy()               # (420  × 7987)
y_np = y_custom.sum(axis=1).to_numpy()  # (7987,)

tier_impacts = {}
x_tier = y_np.copy()

for tier in range(10):
  tier_impacts[tier] = S_np @ x_tier  # emissions at this tier (420,)
  x_tier = A_np @ x_tier              # propagate one step upstream

# Compile: stressors × tiers
tier_df = pd.DataFrame(
  tier_impacts,
  index=exio.air_emissions.S.index
)
tier_df.columns.name = "tier"

# Total air emissions per tier
totals = tier_df.sum(axis=0)
cumulative_pct = totals.cumsum() / totals.sum() * 100

summary = pd.DataFrame({"kg": totals, "cumulative_%": cumulative_pct})
print(summary)


In [ ]:
A_clean = exio.A.fillna(0)
A_np = A_clean.to_numpy()
S_np = S_clean.to_numpy()
y_np = y_custom.sum(axis=1).to_numpy()

stressor_tiers = {}
industry_tiers = {}
x_tier = y_np.copy()

for tier in range(10):
  D_tier = S_np * x_tier          # (420 × 7987): stressor × industry at this tier
  stressor_tiers[tier] = D_tier.sum(axis=1)   # collapse industries → per stressor
  industry_tiers[tier] = D_tier.sum(axis=0)   # collapse stressors  → per industry
  x_tier = A_np @ x_tier          # propagate one step upstream

# --- GHG stressors × tiers ---
stressor_df = pd.DataFrame(stressor_tiers, index=exio.air_emissions.S.index).loc[ghg_stressors]
stressor_df.columns.name = "tier"

# --- (region, sector) × tiers ---
industry_df = pd.DataFrame(industry_tiers, index=exio.L.index)
industry_df.columns.name = "tier"

# Top 20 industries by cumulative impact across all tiers
top_industries = industry_df.sum(axis=1).sort_values(ascending=False).head(20).index

print("=== GHG emissions by stressor × tier (kg) ===")
print(stressor_df[stressor_df.sum(axis=1) > 0].to_string())

print("\n=== Top 20 upstream (region, sector) pairs by tier (kg) ===")
print(industry_df.loc[top_industries].to_string())

In [ ]:
import os
os.makedirs("outputs", exist_ok=True)

# --- Stressor CSV: add total column, drop zero rows, sort by total ---
stressor_out = stressor_df.copy()
stressor_out["total"] = stressor_out.sum(axis=1)
stressor_out = (stressor_out[stressor_out["total"] > 0]
              .sort_values("total", ascending=False))
stressor_out.to_csv("outputs/tier_ghg_stressors.csv")

# --- Industry CSV: sort by total across tiers, drop zero rows ---
industry_total = industry_df.sum(axis=1)
industry_out = industry_df.loc[industry_total[industry_total > 0]
                             .sort_values(ascending=False).index].copy()
industry_out["total"] = industry_out.sum(axis=1)
industry_out.to_csv("outputs/tier_industries.csv")

print(f"tier_ghg_stressors.csv — {stressor_out.shape[0]} stressors × {stressor_df.shape[1]} tiers")
print(f"tier_industries.csv    — {industry_out.shape[0]} industries × {industry_df.shape[1]} tiers")

In [ ]:
industry_total = industry_df.sum(axis=1)
industry_out = (industry_df
              .loc[industry_total[industry_total > 0]
                   .sort_values(ascending=False).index]
              .copy())
industry_out["total"] = industry_out.sum(axis=1)
industry_out = industry_out.reset_index()

industry_out.to_csv("outputs/tier_industries.csv", index=False)
print(f"tier_industries.csv — {len(industry_out)} rows, columns:{industry_out.columns.tolist()}")


In [ ]:
# Compare tier-sum vs. full L-based result
full_footprint = (S_np @ exio.L.fillna(0).to_numpy()) @ y_np
tier_sum_footprint = sum(stressor_tiers.values())  # sum across all 10 tiers

print("Full Leontief footprint (GHG stressors):")
print(pd.Series(full_footprint, index=exio.air_emissions.S.index).loc[ghg_stressors])

print("\n10-tier approximation:")
print(pd.Series(tier_sum_footprint, index=exio.air_emissions.S.index).loc[ghg_stressors])

print("\nCoverage (%):", tier_sum_footprint.sum() / full_footprint.sum() * 100)

In [ ]:
# x_custom already has shape (7987, n_demand_cols)
# Group output requirements by sector across all regions
x_series = x_custom.sum(axis=1)  # already computed as x_vec
x_by_sector = x_series.groupby(level="sector").sum().sort_values(ascending=False)
x_by_region = x_series.groupby(level="region").sum().sort_values(ascending=False)

print("Top 15 sectors by total output triggered:")
print(x_by_sector.head(15))

print("\nTop 15 regions by total output triggered:")
print(x_by_region.head(15))

In [20]:
# ── Monetary inter-industry requirements by (region, sector) × tier ──────────

A_np  = exio.A.fillna(0).to_numpy()   # reuse if already computed
y_np  = y_custom.sum(axis=1).to_numpy()

monetary_tiers = {}
x_tier = y_np.copy()

for tier in range(10):
    monetary_tiers[tier] = x_tier     # output required at this tier (monetary)
    x_tier = A_np @ x_tier            # propagate one step upstream

# Build DataFrame: (region, sector) × tier
monetary_df = pd.DataFrame(
    monetary_tiers,
    index=exio.A.index               # MultiIndex: (region, sector)
)
monetary_df.columns.name = "tier"

# Filter, sort, add total — mirrors your industry_out pattern exactly
monetary_total = monetary_df.sum(axis=1)
monetary_out = (
    monetary_df
    .loc[monetary_total[monetary_total > 0]
         .sort_values(ascending=False).index]
    .copy()
)
monetary_out["total"] = monetary_out.sum(axis=1)
monetary_out = monetary_out.reset_index()

monetary_out.to_csv("outputs/tier_monetary_requirements.csv", index=False)
print(f"tier_monetary_requirements.csv — {len(monetary_out)} rows")
print(monetary_out.head(20).to_string())

tier_monetary_requirements.csv — 6118 rows
tier region                                                                                                                                sector    0         1         2         3         4         5         6         7         8             9     total
0        SE                                                                                                           Real estate activities (70)  1.0  0.010463  0.006859  0.001831  0.000547  0.000179  0.000063  0.000023  0.000009  3.460036e-06  1.019976
1        SE                                                                   Financial intermediation, except insurance and pension funding (65)  0.0  0.083676  0.006572  0.001549  0.000453  0.000152  0.000054  0.000020  0.000008  3.115312e-06  0.092487
2        SE                                                                                                        Other business activities (74)  0.0  0.055014  0.011072  0.003360  0.001031  

In [21]:
x_total = exio.L.fillna(0).to_numpy() @ y_np  # shape (7987,)

monetary_L = pd.Series(x_total, index=exio.A.index)
monetary_L = monetary_L[monetary_L > 0].sort_values(ascending=False).reset_index()
monetary_L.to_csv("outputs/total_monetary_requirements.csv", index=False)